In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
import numpy as np
file_path = "/content/drive/MyDrive/Lion-Biologging-SoS'26/lion_sequences_X_3d.npy"
X_3D = np.load(file_path)
print(X_3D.shape)

(15953, 24, 8)


In [11]:
#our data has some NaN values which have creeped in . we will eliminate them
clean_mask = ~np.isnan(X_3D).any(axis=(1, 2))
X_3D_clean = X_3D[clean_mask]
print(X_3D_clean.shape)

(15952, 24, 8)


In [12]:
#data normalization (scaling all numbers between 0 and 1)
from sklearn.preprocessing import MinMaxScaler

nsamples, ntimesteps, nfeatures = X_3D_clean.shape

X_flat = X_3D_clean.reshape(-1, nfeatures) #scaler reads only 2D matrix (-1 is a dynamic placeholder)

scaler = MinMaxScaler(feature_range=(0,1))
X_flat_scaled = scaler.fit_transform(X_flat)

X_scaled = X_flat_scaled.reshape(nsamples, ntimesteps, nfeatures) #reshaping into 3D

print(f'New matrix shape: {X_scaled.shape}')
print(f"Minimum value in data: {X_scaled.min()}")
print(f"Maximum value in data: {X_scaled.max()}")



New matrix shape: (15952, 24, 8)
Minimum value in data: 0.0
Maximum value in data: 1.0


In [14]:
#defining input layer
from tensorflow.keras.layers import Input

inputs = Input(shape=(ntimesteps, nfeatures), name='Input_Gate')
print(inputs.shape)

(None, 24, 8)


In [22]:
from tensorflow.keras.layers import LSTM

encoder = LSTM(units=16, return_sequences=False, name="LSTM_Encoder")(inputs)

print(encoder.shape)
print(type(encoder))

(None, 16)
<class 'keras.src.backend.common.keras_tensor.KerasTensor'>


In [26]:
#stretching the 16unit bottleneck into 24 for decoder use
from tensorflow.keras.layers import RepeatVector

bridge = RepeatVector(ntimesteps, name='Bridge')(encoder)
print(bridge.shape)


(None, 24, 16)


In [28]:
#setting up decoder
decoder_lstm = LSTM(units=16, return_sequences=True, name='LSTM_Decoder')(bridge)
print(decoder_lstm.shape)

(None, 24, 16)


In [29]:
from tensorflow.keras.layers import TimeDistributed, Dense
from tensorflow.keras.models import Model

#condensing acc to our number of features in our data
outputs = TimeDistributed(Dense(units=nfeatures), name="Output_Gate")(decoder_lstm)

#connecting all layers input to output
lion_autoencoder = Model(inputs=inputs, outputs=outputs, name="Lion_Autoencoder_Model")

In [30]:
#compiling the network with an optimizer and loss function
lion_autoencoder.compile(optimizer='adam', loss='mse')

In [31]:
#training the model on our dataset
history = lion_autoencoder.fit(
    x=X_scaled,           # input: The raw sequences
    y=X_scaled,           # target: Reconstruct the exact same sequence
    epochs=20,            # no of cycle
    batch_size=32,        # 32 day chunking
    validation_split=0.1, # holding back 10% data for later validation
    shuffle=True          # mixing up
)

Epoch 1/20
449/449 ━━━━━━━━━━━━━━━━━━━━ 15s 21ms/step - loss: 0.0980 - val_loss: 0.0719
Epoch 2/20
449/449 ━━━━━━━━━━━━━━━━━━━━ 11s 24ms/step - loss: 0.0644 - val_loss: 0.0633
Epoch 3/20
449/449 ━━━━━━━━━━━━━━━━━━━━ 11s 24ms/step - loss: 0.0619 - val_loss: 0.0623
Epoch 4/20
449/449 ━━━━━━━━━━━━━━━━━━━━ 20s 22ms/step - loss: 0.0609 - val_loss: 0.0617
Epoch 5/20
449/449 ━━━━━━━━━━━━━━━━━━━━ 12s 27ms/step - loss: 0.0605 - val_loss: 0.0614
Epoch 6/20
449/449 ━━━━━━━━━━━━━━━━━━━━ 10s 23ms/step - loss: 0.0602 - val_loss: 0.0612
Epoch 7/20
449/449 ━━━━━━━━━━━━━━━━━━━━ 9s 20ms/step - loss: 0.0599 - val_loss: 0.0608
Epoch 8/20
449/449 ━━━━━━━━━━━━━━━━━━━━ 11s 24ms/step - loss: 0.0595 - val_loss: 0.0605
Epoch 9/20
449/449 ━━━━━━━━━━━━━━━━━━━━ 19s 20ms/step - loss: 0.0591 - val_loss: 0.0604
Epoch 10/20
449/449 ━━━━━━━━━━━━━━━━━━━━ 12s 23ms/step - loss: 0.0589 - val_loss: 0.0602
Epoch 11/20
449/449 ━━━━━━━━━━━━━━━━━━━━ 11s 24ms/step - loss: 0.0588 - val_loss: 0.0601
Epoch 12/20
449/449 ━━━━━━━━━━━

In [34]:
#saving our model and the clean 3D file
model_path = "/content/drive/MyDrive/Lion-Biologging-SoS'26/models/autoencoder_v1.keras"
lion_autoencoder.save(model_path)

np.save("/content/drive/MyDrive/Lion-Biologging-SoS'26/X_scaled_clean.npy", X_scaled)